In [ ]:
# =============================================================================
# CNN for Bug Severity Text Classification
# Pretrained embedding: GloVe 300d
# Preprocessing: lowercase, remove punctuation, stopwords, lemmatization
# Checkpoint: best Validation Loss + best Macro F1 (saved separately)
# =============================================================================

import os
import re
import string
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)

# 0. LOGGING SETUP

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("training_cnn.log", mode="w"),
    ],
)
logger = logging.getLogger(__name__)

# 1. CONFIG

In [ ]:
class Config:
    # Data
    # Chọn uncomment 1 trong các file sau để chọn training set tương ứng cho từng thực nghiệm

    TRAIN_PATH = "/root/train.csv"                              # w/o augmentation
    # TRAIN_PATH = "/root/train_eda.csv"                          # EDA augmentation
    # TRAIN_PATH = "/root/train_aeda.csv"                         # AEDA augmentation
    # TRAIN_PATH = "/root/train_ease.csv"                         # EASE augmentation
    # TRAIN_PATH = "/root/train-llm-aug-gpt-5-2025-08-07.csv"     # LLM augmentation (gpt-5-2025-08-07)

    VAL_PATH   = "/root/val.csv"
    TEST_PATH  = "/root/test.csv"

    # GloVe – download & place at this path, or update accordingly
    # Download: https://nlp.stanford.edu/data/glove.6B.zip  (glove.6B.300d.txt)
    GLOVE_PATH = "/root/glove.6B.300d.txt"

    # Checkpoints
    CKPT_DIR         = "./checkpoints"
    CKPT_BEST_LOSS   = "best_val_loss.pt"
    CKPT_BEST_F1     = "best_macro_f1.pt"

    # Text
    MAX_VOCAB_SIZE = 50_000
    MAX_SEQ_LEN    = 128
    EMBED_DIM      = 300        # must match GloVe file

    # CNN architecture
    NUM_FILTERS    = 128        # number of filters per kernel size
    KERNEL_SIZES   = [2, 3, 4, 5]
    DROPOUT        = 0.5

    # Training
    NUM_CLASSES    = 5
    BATCH_SIZE     = 128
    LR             = 1e-3
    WEIGHT_DECAY   = 1e-4
    NUM_EPOCHS     = 50
    EARLY_STOP_PATIENCE = 5     # stop after N epochs with no val-loss improvement
    SCHEDULER_PATIENCE  = 2     # reduce LR after N epochs with no val-loss improvement

    SEED    = 42
    DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# reproducibility
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

# 2. LABEL MAPPING

In [ ]:
LABEL_MAP = {
    "Trivial":  0,
    "Minor":    1,
    "Major":    2,
    "Critical": 3,
    "Blocker":  4,
}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

# 3. DATA LOADING  (with optional class-balancing)

In [ ]:
def load_data():
    df_train = pd.read_csv(cfg.TRAIN_PATH)
    df_val   = pd.read_csv(cfg.VAL_PATH)
    df_test  = pd.read_csv(cfg.TEST_PATH)
    return df_train, df_val, df_test


# ===== UNDER-SAMPLING WITH REAL TRAINING DATA =====
def process_training_set_with_balance_class_distribution(dataframe, target_column_name="priority"):
    min_sample = dataframe[target_column_name].value_counts().min()
    df_filtered = dataframe.groupby(target_column_name, group_keys=False).apply(
        lambda x: x.sample(n=min_sample, random_state=42)
    ).reset_index(drop=True)
    return df_filtered


# ===== OVER-SAMPLING WITH REAL TRAINING DATA =====
def process_training_set_with_oversampling(dataframe, target_column_name="priority"):
    max_sample = dataframe[target_column_name].value_counts().max()
    df_oversampled = dataframe.groupby(target_column_name, group_keys=False).apply(
        lambda x: x.sample(n=max_sample, replace=True, random_state=42)
    ).reset_index(drop=True)
    return df_oversampled

# 4. TEXT PREPROCESSING

In [ ]:
_lemmatizer  = WordNetLemmatizer()
_stop_words  = set(stopwords.words("english"))
_punct_table = str.maketrans("", "", string.punctuation)


def preprocess_text(text: str) -> str:
    """Lowercase → remove punctuation → remove stopwords → lemmatize."""
    if not isinstance(text, str):
        text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation
    text = text.translate(_punct_table)

    # 3. Tokenize (simple whitespace split after cleaning)
    tokens = text.split()

    # 4. Remove stopwords
    tokens = [t for t in tokens if t not in _stop_words]

    # 5. Lemmatize
    tokens = [_lemmatizer.lemmatize(t) for t in tokens]

    return " ".join(tokens)

# 5. VOCABULARY

In [ ]:
class Vocabulary:
    PAD_TOKEN = "<PAD>"
    UNK_TOKEN = "<UNK>"

    def __init__(self, max_size: int = cfg.MAX_VOCAB_SIZE):
        self.max_size  = max_size
        self.token2idx = {}
        self.idx2token = {}
        self._freq     = {}

    def build(self, texts):
        """Count token frequencies from a list of (already-preprocessed) strings."""
        for text in texts:
            for tok in text.split():
                self._freq[tok] = self._freq.get(tok, 0) + 1

        # Keep top-N by frequency
        sorted_tokens = sorted(self._freq, key=self._freq.get, reverse=True)
        vocab_tokens  = sorted_tokens[: self.max_size - 2]   # reserve PAD & UNK

        self.token2idx = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.token2idx.update({tok: i + 2 for i, tok in enumerate(vocab_tokens)})
        self.idx2token = {v: k for k, v in self.token2idx.items()}
        logger.info(f"Vocabulary built: {len(self.token2idx):,} tokens")

    def encode(self, text: str, max_len: int) -> list:
        """Convert text → padded/truncated list of indices."""
        tokens = text.split()[:max_len]
        ids    = [self.token2idx.get(t, 1) for t in tokens]   # 1 = UNK
        ids   += [0] * (max_len - len(ids))                    # 0 = PAD
        return ids

    def __len__(self):
        return len(self.token2idx)

# 6. GLOVE EMBEDDING MATRIX

In [ ]:
def load_glove_embeddings(vocab: Vocabulary, glove_path: str, embed_dim: int):
    """Build an embedding matrix aligned with vocab indices."""
    logger.info(f"Loading GloVe from: {glove_path}")
    glove = {}
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Reading GloVe"):
            parts = line.rstrip().split(" ")
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            glove[word] = vec

    vocab_size = len(vocab)
    matrix     = np.zeros((vocab_size, embed_dim), dtype=np.float32)

    hits = 0
    for token, idx in vocab.token2idx.items():
        if token in glove:
            matrix[idx] = glove[token]
            hits += 1
        else:
            # Random init for unknown tokens (PAD stays zero)
            if token != vocab.PAD_TOKEN:
                matrix[idx] = np.random.normal(0, 0.1, embed_dim)

    logger.info(
        f"GloVe coverage: {hits:,}/{vocab_size:,} tokens "
        f"({100*hits/vocab_size:.1f}%)"
    )
    return torch.tensor(matrix)

# 7. DATASET

In [ ]:
class BugReportDataset(Dataset):
    def __init__(self, texts, labels, vocab: Vocabulary, max_len: int):
        self.encodings = [vocab.encode(t, max_len) for t in texts]
        self.labels    = labels.tolist() if hasattr(labels, "tolist") else list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encodings[idx], dtype=torch.long),
            torch.tensor(self.labels[idx],    dtype=torch.long),
        )

# 8. DEFINE CNN MODEL  (TextCNN)

In [ ]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size:    int,
        embed_dim:     int,
        num_classes:   int,
        num_filters:   int,
        kernel_sizes:  list,
        dropout:       float,
        pretrained_embeddings: torch.Tensor = None,
        freeze_embeddings: bool = False,
    ):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )
        if pretrained_embeddings is not None:
            self.embedding.weight = nn.Parameter(pretrained_embeddings)
        if freeze_embeddings:
            self.embedding.weight.requires_grad = False

        # Parallel convolutional blocks (one per kernel size)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(
                    in_channels  = embed_dim,
                    out_channels = num_filters,
                    kernel_size  = k,
                    padding      = k // 2,      # same-ish padding
                ),
                nn.BatchNorm1d(num_filters),
                nn.ReLU(),
            )
            for k in kernel_sizes
        ])

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)          # (batch, seq_len, embed_dim)
        emb = emb.permute(0, 2, 1)       # (batch, embed_dim, seq_len)

        pooled = []
        for conv in self.convs:
            c = conv(emb)                # (batch, num_filters, seq_len')
            p = F.adaptive_max_pool1d(c, 1).squeeze(-1)  # (batch, num_filters)
            pooled.append(p)

        out = torch.cat(pooled, dim=1)   # (batch, num_filters * len(kernels))
        out = self.dropout(out)
        return self.classifier(out)      # (batch, num_classes)

# 9. TRAIN / EVAL HELPERS

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    """One epoch. If optimizer is None → eval mode."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            logits = model(texts)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss    += loss.item() * labels.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, macro_f1

# 10. TRAINING LOOP

In [ ]:
def train(model, train_loader, val_loader, cfg):
    device    = cfg.DEVICE
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5,
        patience=cfg.SCHEDULER_PATIENCE,
    )

    best_val_loss = float("inf")
    best_macro_f1 = 0.0
    no_improve    = 0

    logger.info("=" * 65)
    logger.info(
        f"{'Epoch':>5} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | "
        f"{'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7}"
    )
    logger.info("=" * 65)

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = run_epoch(
            model, train_loader, criterion, optimizer, device
        )
        va_loss, va_acc, va_f1 = run_epoch(
            model, val_loader, criterion, None, device
        )

        scheduler.step(va_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        logger.info(
            f"{epoch:>5} | {tr_loss:>8.4f} | {tr_acc:>7.4f} | {tr_f1:>7.4f} | "
            f"{va_loss:>8.4f} | {va_acc:>7.4f} | {va_f1:>7.4f} | lr={current_lr:.2e}"
        )

        # ── Save best by Validation Loss ──────────────────────────────────
        if va_loss < best_val_loss:
            best_val_loss = va_loss
            no_improve    = 0
            ckpt_path = os.path.join(cfg.CKPT_DIR, cfg.CKPT_BEST_LOSS)
            torch.save(model.state_dict(), ckpt_path)
            logger.info(
                f"  ✔ New best val-loss={best_val_loss:.4f} → saved to {ckpt_path}"
            )
        else:
            no_improve += 1

        # ── Save best by Macro F1 ─────────────────────────────────────────
        if va_f1 > best_macro_f1:
            best_macro_f1 = va_f1
            ckpt_path = os.path.join(cfg.CKPT_DIR, cfg.CKPT_BEST_F1)
            torch.save(model.state_dict(), ckpt_path)
            logger.info(
                f"  ✔ New best macro-f1={best_macro_f1:.4f} → saved to {ckpt_path}"
            )

        # ── Early stopping (tracks val-loss) ──────────────────────────────
        if no_improve >= cfg.EARLY_STOP_PATIENCE:
            logger.info(
                f"\n⚠  Early stopping triggered after {epoch} epochs "
                f"(no val-loss improvement for {cfg.EARLY_STOP_PATIENCE} epochs)."
            )
            break

    logger.info("=" * 65)
    logger.info(
        f"Training finished | best val-loss={best_val_loss:.4f} | "
        f"best macro-f1={best_macro_f1:.4f}"
    )
    return model

# 11. EVALUATION ON TEST SET

In [ ]:
def evaluate_on_test(model, test_loader, checkpoint_path, cfg):
    logger.info(f"\nLoading checkpoint: {checkpoint_path}")
    model.load_state_dict(
        torch.load(checkpoint_path, map_location=cfg.DEVICE)
    )
    model.eval()
    model.to(cfg.DEVICE)

    criterion = nn.CrossEntropyLoss()
    te_loss, _, _ = run_epoch(model, test_loader, criterion, None, cfg.DEVICE)

    # Collect all predictions and labels
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in test_loader:
            texts = texts.to(cfg.DEVICE)
            preds = model(texts).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ── Core metrics (matching your evaluation format) ────────────────────
    accuracy       = round(accuracy_score(all_labels, all_preds) * 100, 2)
    precision_micro = round(precision_score(all_labels, all_preds, average="micro", zero_division=0) * 100, 2)
    recall_micro    = round(recall_score(all_labels, all_preds,    average="micro", zero_division=0) * 100, 2)
    f1_micro        = round(f1_score(all_labels, all_preds,        average="micro", zero_division=0) * 100, 2)

    # ── Extra macro / weighted metrics ───────────────────────────────────
    precision_macro    = round(precision_score(all_labels, all_preds, average="macro",    zero_division=0) * 100, 2)
    recall_macro       = round(recall_score(all_labels, all_preds,    average="macro",    zero_division=0) * 100, 2)
    f1_macro           = round(f1_score(all_labels, all_preds,        average="macro",    zero_division=0) * 100, 2)
    precision_weighted = round(precision_score(all_labels, all_preds, average="weighted", zero_division=0) * 100, 2)
    recall_weighted    = round(recall_score(all_labels, all_preds,    average="weighted", zero_division=0) * 100, 2)
    f1_weighted        = round(f1_score(all_labels, all_preds,        average="weighted", zero_division=0) * 100, 2)

    # ── Per-class report & confusion matrix ───────────────────────────────
    target_names = [ID2LABEL[i] for i in range(cfg.NUM_CLASSES)]
    report = classification_report(
        all_labels, all_preds,
        target_names=target_names, zero_division=0,
    )
    cm = confusion_matrix(all_labels, all_preds)

    ckpt_name = os.path.basename(checkpoint_path)
    logger.info(f"\n{'='*55}")
    logger.info(f"Test Results — checkpoint: {ckpt_name}")
    logger.info(f"{'='*55}")
    logger.info(f"Loss                : {te_loss:.4f}")
    logger.info(f"Accuracy            : {accuracy}")
    logger.info(f"Precision Macro     : {precision_macro}")
    logger.info(f"Recall    Macro     : {recall_macro}")
    logger.info(f"F1 score  Macro     : {f1_macro}")
    logger.info(f"Precision Micro     : {precision_micro}")
    logger.info(f"Recall    Micro     : {recall_micro}")
    logger.info(f"F1 score  Micro     : {f1_micro}")
    logger.info(f"Precision Weighted  : {precision_weighted}")
    logger.info(f"Recall    Weighted  : {recall_weighted}")
    logger.info(f"F1 score  Weighted  : {f1_weighted}")
    logger.info(f"\nClassification Report:\n{report}")
    logger.info(f"Confusion Matrix:\n{cm}")

    return {
        "loss"              : te_loss,
        "accuracy"          : accuracy,
        "precision_micro"   : precision_micro,
        "recall_micro"      : recall_micro,
        "f1_micro"          : f1_micro,
        "precision_macro"   : precision_macro,
        "recall_macro"      : recall_macro,
        "f1_macro"          : f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted"   : recall_weighted,
        "f1_weighted"       : f1_weighted,
        "report"            : report,
        "confusion_matrix"  : cm,
    }

# 12. MAIN

In [ ]:
def main():
    logger.info(f"Device: {cfg.DEVICE}")

    # ── Load raw data ─────────────────────────────────────────────────────
    df_train, df_val, df_test = load_data()
    print("=============================================================================")
    lengths = df_train["text"].apply(lambda x: len(preprocess_text(x).split()))
    print(lengths.describe())
    print(f"90th percentile: {lengths.quantile(0.90):.0f} tokens")
    print(f"95th percentile: {lengths.quantile(0.95):.0f} tokens")
    print(f"99th percentile: {lengths.quantile(0.99):.0f} tokens")
    print("=============================================================================")

    # ── (Optional) CLASS BALANCING PROCESSING – UNCOMMENT 1 BLOCK FOR SPECIFIC SAMPLING METHOD OR COMMENT BOTH FOR SKIP SAMPLING METHOD ─────────────────
    # df_train = process_training_set_with_balance_class_distribution(df_train)
    # df_train = process_training_set_with_oversampling(df_train)

    logger.info(
        f"Sizes → train: {len(df_train):,} | val: {len(df_val):,} | "
        f"test: {len(df_test):,}"
    )
    logger.info(f"Train class dist:\n{df_train['priority'].value_counts()}")

    # ── Labels ────────────────────────────────────────────────────────────
    y_train = df_train["priority"].map(LABEL_MAP)
    y_val   = df_val["priority"].map(LABEL_MAP)
    y_test  = df_test["priority"].map(LABEL_MAP)

    # ── Preprocessing ─────────────────────────────────────────────────────
    logger.info("Preprocessing texts …")
    train_texts = df_train["text"].apply(preprocess_text).tolist()
    val_texts   = df_val["text"].apply(preprocess_text).tolist()
    test_texts  = df_test["text"].apply(preprocess_text).tolist()

    # ── Vocabulary (built on train only) ──────────────────────────────────
    vocab = Vocabulary(max_size=cfg.MAX_VOCAB_SIZE)
    vocab.build(train_texts)

    # ── GloVe embedding matrix ────────────────────────────────────────────
    embed_matrix = load_glove_embeddings(vocab, cfg.GLOVE_PATH, cfg.EMBED_DIM)

    # ── Datasets & DataLoaders ────────────────────────────────────────────
    train_ds = BugReportDataset(train_texts, y_train, vocab, cfg.MAX_SEQ_LEN)
    val_ds   = BugReportDataset(val_texts,   y_val,   vocab, cfg.MAX_SEQ_LEN)
    test_ds  = BugReportDataset(test_texts,  y_test,  vocab, cfg.MAX_SEQ_LEN)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True
    )
    val_loader   = DataLoader(
        val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )
    test_loader  = DataLoader(
        test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )

    # ── Model ─────────────────────────────────────────────────────────────
    model = TextCNN(
        vocab_size            = len(vocab),
        embed_dim             = cfg.EMBED_DIM,
        num_classes           = cfg.NUM_CLASSES,
        num_filters           = cfg.NUM_FILTERS,
        kernel_sizes          = cfg.KERNEL_SIZES,
        dropout               = cfg.DROPOUT,
        pretrained_embeddings = embed_matrix,
        freeze_embeddings     = False,   # set True to freeze GloVe weights
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model parameters: {n_params:,}")

    # ── Training ──────────────────────────────────────────────────────────
    train(model, train_loader, val_loader, cfg)

    # ── Test evaluation with BOTH checkpoints ─────────────────────────────
    logger.info("\n" + "=" * 65)
    logger.info("EVALUATING ON TEST SET")
    logger.info("=" * 65)

    # 1) Best by Validation Loss
    evaluate_on_test(
        model, test_loader,
        checkpoint_path=os.path.join(cfg.CKPT_DIR, cfg.CKPT_BEST_LOSS),
        cfg=cfg,
    )

    # 2) Best by Macro F1
    evaluate_on_test(
        model, test_loader,
        checkpoint_path=os.path.join(cfg.CKPT_DIR, cfg.CKPT_BEST_F1),
        cfg=cfg,
    )


if __name__ == "__main__":
    main()

2026-03-16 15:31:24,265 [INFO] Device: cuda


/tmp/ipykernel_928/523276837.py:134: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_oversampled = dataframe.groupby(target_column_name, group_keys=False).apply(
2026-03-16 15:31:32,582 [INFO] Sizes → train: 93,425 | val: 400 | test: 400
2026-03-16 15:31:32,589 [INFO] Train class dist:
priority
Blocker     18685
Critical    18685
Major       18685
Minor       18685
Trivial     18685
Name: count, dtype: int64
2026-03-16 15:31:32,595 [INFO] Preprocessing texts …


count    28755.000000
mean        52.541958
std         67.045153
min          2.000000
25%         20.000000
50%         34.000000
75%         61.000000
max       1065.000000
Name: text, dtype: float64
90th percentile: 108 tokens
95th percentile: 150 tokens
99th percentile: 303 tokens


2026-03-16 15:31:50,023 [INFO] Vocabulary built: 33,378 tokens
2026-03-16 15:31:50,024 [INFO] Loading GloVe from: /kaggle/input/datasets/aellatif/glove6b300dtxt/glove.6B.300d.txt
Reading GloVe: 400000it [00:23, 16937.83it/s]
2026-03-16 15:32:13,960 [INFO] GloVe coverage: 6,372/33,378 tokens (19.1%)
2026-03-16 15:32:15,134 [INFO] Model parameters: 10,555,101
2026-03-16 15:32:16,475 [INFO] =================================================================
2026-03-16 15:32:16,476 [INFO] Epoch |   TrLoss |   TrAcc |    TrF1 |   VaLoss |   VaAcc |    VaF1
2026-03-16 15:32:16,476 [INFO] =================================================================
2026-03-16 15:32:30,457 [INFO]     1 |   0.2275 |  0.9206 |  0.9204 |   1.7289 |  0.6275 |  0.2526 | lr=1.00e-03
2026-03-16 15:32:30,560 [INFO]   ✔ New best val-loss=1.7289 → saved to ./checkpoints/best_val_loss.pt
2026-03-16 15:32:30,655 [INFO]   ✔ New best macro-f1=0.2526 → saved to ./checkpoints/best_macro_f1.pt
2026-03-16 15:32:43,940 [INFO]